In [2]:
pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [4]:
spark=SparkSession.builder.appName("Transforms").getOrCreate()

In [94]:
from pyarrow import null
banking_data = [
    (1001, "C101", "Savings", "Deposit", 50000.0, "Hyderabad", "2026-01-05"),
    (1002, "C102", "Current", "Withdrawal", 12000.0, "Bengaluru", "2026-01-07"),
    (1003, "C103", "Savings", "Transfer", 25000.0, "Chennai", "2026-01-10"),
    (1004, "C104", "Salary", "Deposit", 75000.0, "Mumbai", "2026-01-12"),
    (1005, "C105", "Savings", "Withdrawal", 8000.0, "Delhi", "2026-01-15"),
    (1006, "C101", "Savings", "Transfer", 15000.0, "Hyderabad", "2026-01-18"),
    (1007, "C106", "Current", "Deposit", 90000.0, "Pune", "2026-01-20"),
    (1008, "C107", "Savings", "Withdrawal", 5000.0, "Kolkata", "2026-01-22"),
    (1009, "C108", "Salary", "Deposit", 65000.0, "Bengaluru", "2026-01-25"),
    (1010, "C102", "Current", "Transfer", 30000.0, "Bengaluru", "2026-01-28"),
    (1011, "C109", "Savings", "Deposit", 40000.0, "Chennai", "2026-02-02"),
    (1012, "C110", "Current", "Withdrawal", 18000.0, "Mumbai", "2026-02-05"),
    (1013, "C103", "Savings", "Transfer", 22000.0, "Chennai", "2026-02-08"),
    (1014, "C111", "Salary", "Deposit", 85000.0, "Delhi", "2026-02-10"),
    (1015, "C105", "Savings", "Withdrawal", 7000.0, "Hyderabad", "2026-02-12"),
    (1001, "C106", "Savings", "Deposit", 50000.0, "Hyderabad", "2026-01-05"),
    (1016, "C106", "Savings", "Deposit", None, "Hyderabad", "2026-01-05"),
    (1017, "C106", "Savings", "Deposit", 50000.0, None, "2026-01-05")




]
cols=["transaction_id","customer_id","account_type","transaction_type","amount","city","transaction_date"]
df=spark.createDataFrame(banking_data,cols)


1. Select transaction ID, customer ID and amount.


In [48]:
df.select("transaction_id","customer_id","amount").show()

+--------------+-----------+-------+
|transaction_id|customer_id| amount|
+--------------+-----------+-------+
|          1001|       C101|50000.0|
|          1002|       C102|12000.0|
|          1003|       C103|25000.0|
|          1004|       C104|75000.0|
|          1005|       C105| 8000.0|
|          1006|       C101|15000.0|
|          1007|       C106|90000.0|
|          1008|       C107| 5000.0|
|          1009|       C108|65000.0|
|          1010|       C102|30000.0|
|          1011|       C109|40000.0|
|          1012|       C110|18000.0|
|          1013|       C103|22000.0|
|          1014|       C111|85000.0|
|          1015|       C105| 7000.0|
|          1001|       C106|50000.0|
|          1016|       C106|   NULL|
|          1017|       C106|50000.0|
+--------------+-----------+-------+



2. Filter transactions above ₹50,000.


In [49]:
df.filter("amount>50000").show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|
|          1007|       C106|     Current|         Deposit|90000.0|     Pune|      2026-01-20|
|          1009|       C108|      Salary|         Deposit|65000.0|Bengaluru|      2026-01-25|
|          1014|       C111|      Salary|         Deposit|85000.0|    Delhi|      2026-02-10|
+--------------+-----------+------------+----------------+-------+---------+----------------+



3. Filter only `DEBIT` transactions.


In [50]:
debit_df=df.filter(col("transaction_type")=="Withdrawal")
debit_df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|      2026-01-15|
|          1008|       C107|     Savings|      Withdrawal| 5000.0|  Kolkata|      2026-01-22|
|          1012|       C110|     Current|      Withdrawal|18000.0|   Mumbai|      2026-02-05|
|          1015|       C105|     Savings|      Withdrawal| 7000.0|Hyderabad|      2026-02-12|
+--------------+-----------+------------+----------------+-------+---------+----------------+



4. Create `transaction_fee` as 1% of amount.


In [51]:
df=df.withColumn("transaction_fee",col("amount")*0.01)
df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|          750.0|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|      2026-01-15|           80.0|
|          1006|       C101|     Savings|        Transfer|15000.0|Hyderabad|      2026-01-18|          150.0|
|         

5. Create `net_amount` after fee.


In [52]:
# df=df.withColumn("net_amount",col("amount")*1.01)
df=df.withColumn("net_amount",col("amount")-col("transaction_fee"))
df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|net_amount|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|   49500.0|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|   11880.0|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|   24750.0|
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|          750.0|   74250.0|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|      2026-01-15|           80.0|    7920.0|
|          1006|       C101|    

6. Rename `city` to `branch_city`.


In [53]:
# df.withColumnRenamed("old_name","new_name")
df.withColumnRenamed("city","branch_city").show()

+--------------+-----------+------------+----------------+-------+-----------+----------------+---------------+----------+
|transaction_id|customer_id|account_type|transaction_type| amount|branch_city|transaction_date|transaction_fee|net_amount|
+--------------+-----------+------------+----------------+-------+-----------+----------------+---------------+----------+
|          1001|       C101|     Savings|         Deposit|50000.0|  Hyderabad|      2026-01-05|          500.0|   49500.0|
|          1002|       C102|     Current|      Withdrawal|12000.0|  Bengaluru|      2026-01-07|          120.0|   11880.0|
|          1003|       C103|     Savings|        Transfer|25000.0|    Chennai|      2026-01-10|          250.0|   24750.0|
|          1004|       C104|      Salary|         Deposit|75000.0|     Mumbai|      2026-01-12|          750.0|   74250.0|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|      Delhi|      2026-01-15|           80.0|    7920.0|
|          1006|

7. Drop an unnecessary column.


In [12]:
df.drop("net_amount").show()

+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|          750.0|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|      2026-01-15|           80.0|
|          1006|       C101|     Savings|        Transfer|15000.0|Hyderabad|      2026-01-18|          150.0|
|         

8. Find distinct account types.


In [13]:
df.select("account_type").distinct().show()

+------------+
|account_type|
+------------+
|     Savings|
|     Current|
|      Salary|
+------------+



9. Remove duplicate transaction IDs.


In [14]:
df.select("transaction_id").drop_duplicates().show()

+--------------+
|transaction_id|
+--------------+
|          1002|
|          1009|
|          1007|
|          1005|
|          1001|
|          1008|
|          1004|
|          1006|
|          1003|
|          1010|
|          1016|
|          1012|
|          1013|
|          1011|
|          1015|
|          1014|
|          1017|
+--------------+



10. Sort transactions by amount descending.


In [15]:
df.select("amount").sort("amount",ascending=False).show()

+-------+
| amount|
+-------+
|90000.0|
|85000.0|
|75000.0|
|65000.0|
|50000.0|
|50000.0|
|50000.0|
|40000.0|
|30000.0|
|25000.0|
|22000.0|
|18000.0|
|15000.0|
|12000.0|
| 8000.0|
| 7000.0|
| 5000.0|
|   NULL|
+-------+



11. Group by account type and calculate count, total amount, average amount, min and max.

In [16]:
df.select("account_type","amount").groupBy("account_type").count().show()
# df.select("account_type","amount").groupBy("account_type").agg(sum("amount")).show()
df.selectExpr("sum(amount) as total_amount").show()
df.selectExpr("avg(amount) as average_amount").show()
df.selectExpr("min(amount) as minimum_amount").show()
df.selectExpr("max(amount) as maximum_amount").show()



+------------+-----+
|account_type|count|
+------------+-----+
|     Savings|   11|
|     Current|    4|
|      Salary|    3|
+------------+-----+

+------------+
|total_amount|
+------------+
|    647000.0|
+------------+

+-----------------+
|   average_amount|
+-----------------+
|38058.82352941176|
+-----------------+

+--------------+
|minimum_amount|
+--------------+
|        5000.0|
+--------------+

+--------------+
|maximum_amount|
+--------------+
|       90000.0|
+--------------+



12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.


In [17]:
df.withColumn("risk_level",when (col("amount")>=100000 ,"High").when(col("amount")>=50000,"Medium").otherwise("low")).show()

+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|net_amount|risk_level|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|   50500.0|    Medium|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|   12120.0|       low|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|   25250.0|       low|
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|          750.0|   75750.0|    Medium|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|

13. Convert transaction date to DateType and extract year/month/day.


In [18]:
import datetime

In [19]:
df=df.withColumn("transaction_date",to_date("transaction_date"))
# df.show()
# df.printSchema()
df=df.withColumn("transaction_year",year(col("transaction_date")))
# df_year=df.selectExpr("extract(year from transaction_date)")
# df.show()
df=df.withColumn("transaction_month",month(col("transaction_date")))
# df.show()
df=df.withColumn("transaction_day",day(col("transaction_date")))
df.show()

+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------------+-----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|net_amount|transaction_year|transaction_month|transaction_day|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------------+-----------------+---------------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|   50500.0|            2026|                1|              5|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|   12120.0|            2026|                1|              7|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|   25250.0|            2026|   

14. Handle null amount/city values appropriately.


In [20]:
avg_value=df.selectExpr("round(avg(amount),1) as avg_value").first()["avg_value"]
print(avg_value)
df=df.fillna(avg_value,subset=["amount"])
df=df.withColumn("amount",round(("amount"),1))
# df.show()
count_city=df.groupBy("city").count().orderBy("count",ascending=False).first()["city"]
print(count_city)
df=df.fillna(count_city,"city")
df.show()

38058.8
Hyderabad
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------------+-----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|transaction_fee|net_amount|transaction_year|transaction_month|transaction_day|
+--------------+-----------+------------+----------------+-------+---------+----------------+---------------+----------+----------------+-----------------+---------------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|          500.0|   50500.0|            2026|                1|              5|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|          120.0|   12120.0|            2026|                1|              7|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|          250.0|   25250.0|  

In [21]:
df.drop("")

DataFrame[transaction_id: bigint, customer_id: string, account_type: string, transaction_type: string, amount: double, city: string, transaction_date: date, transaction_fee: double, net_amount: double, transaction_year: int, transaction_month: int, transaction_day: int]

In [26]:
banking_data_2 = [

    (1016, "C112", "Savings", "Deposit", 45000.0, "Bengaluru", "2026-02-15"),

    (1017, "C113", "Current", "Withdrawal", 10000.0, "Hyderabad", "2026-02-16"),

    (1018, "C114", "Savings", "Transfer", 28000.0, "Chennai", "2026-02-17"),

    (1019, "C115", "Salary", "Deposit", 70000.0, "Pune", "2026-02-18"),

    (1020, "C116", "Current", "Withdrawal", 15000.0, "Mumbai", "2026-02-19"),

    (1021, "C112", "Savings", "Transfer", 12000.0, "Bengaluru", "2026-02-20"),

    (1022, "C117", "Current", "Deposit", 55000.0, "Delhi", "2026-02-21"),

    (1023, "C118", "Savings", "Withdrawal", 6500.0, "Kolkata", "2026-02-22"),

    (1024, "C119", "Salary", "Deposit", 80000.0, "Hyderabad", "2026-02-23"),

    (1025, "C120", "Current", "Transfer", 35000.0, "Chennai", "2026-02-24"),

    (1026, "C113", "Savings", "Deposit", 30000.0, "Hyderabad", "2026-02-25"),

    (1027, "C121", "Current", "Withdrawal", 20000.0, "Pune", "2026-02-26"),

    (1028, "C114", "Savings", "Transfer", 18000.0, "Chennai", "2026-02-27"),

    (1029, "C122", "Salary", "Deposit", 90000.0, "Mumbai", "2026-02-28"),

    (1030, "C116", "Savings", "Withdrawal", 9000.0, "Delhi", "2026-03-01")

]

cols=["transaction_id","customer_id","account_type","transaction_type","amount","city","transaction_date"]
df2=spark.createDataFrame(banking_data_2,cols)
df2.show()
# df2=df2.withColumn("transaction_date")

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          1016|       C112|     Savings|         Deposit|45000.0|Bengaluru|      2026-02-15|
|          1017|       C113|     Current|      Withdrawal|10000.0|Hyderabad|      2026-02-16|
|          1018|       C114|     Savings|        Transfer|28000.0|  Chennai|      2026-02-17|
|          1019|       C115|      Salary|         Deposit|70000.0|     Pune|      2026-02-18|
|          1020|       C116|     Current|      Withdrawal|15000.0|   Mumbai|      2026-02-19|
|          1021|       C112|     Savings|        Transfer|12000.0|Bengaluru|      2026-02-20|
|          1022|       C117|     Current|         Deposit|55000.0|    Delhi|      2026-02-21|
|          1023|       C118|     Savings|      Withdrawal| 6

15. Create another DataFrame and perform `unionByName`.

In [27]:
df_unioun=df.unionByName(df2)
df_unioun.show(50)

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|          1001|       C101|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|
|          1002|       C102|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|
|          1003|       C103|     Savings|        Transfer|25000.0|  Chennai|      2026-01-10|
|          1004|       C104|      Salary|         Deposit|75000.0|   Mumbai|      2026-01-12|
|          1005|       C105|     Savings|      Withdrawal| 8000.0|    Delhi|      2026-01-15|
|          1006|       C101|     Savings|        Transfer|15000.0|Hyderabad|      2026-01-18|
|          1007|       C106|     Current|         Deposit|90000.0|     Pune|      2026-01-20|
|          1008|       C107|     Savings|      Withdrawal| 5

16. Create a customer master DataFrame and join it with transactions.


In [25]:
customer_master_data = [

    ("C101", "Ram", "Sharma", 28, "Male", "Hyderabad", "2022-01-15"),

    ("C102", "Priya", "Reddy", 32, "Female", "Bengaluru", "2021-03-20"),

    ("C103", "Arjun", "Kumar", 35, "Male", "Chennai", "2020-06-10"),

    ("C104", "Sneha", "Patel", 29, "Female", "Mumbai", "2023-02-05"),

    ("C105", "Rahul", "Verma", 41, "Male", "Delhi", "2019-08-18"),

    ("C106", "Anjali", "Shah", 27, "Female", "Pune", "2022-11-12"),

    ("C107", "Vikram", "Singh", 38, "Male", "Kolkata", "2018-05-25"),

    ("C108", "Kavya", "Rao", 31, "Female", "Bengaluru", "2021-09-14"),

    ("C109", "Rohit", "Mehta", 45, "Male", "Chennai", "2017-04-22"),

    ("C110", "Neha", "Gupta", 34, "Female", "Mumbai", "2020-12-01"),

    ("C111", "Kiran", "Reddy", 30, "Male", "Delhi", "2023-07-19"),

    ("C112", "Pooja", "Nair", 26, "Female", "Bengaluru", "2024-01-10"),

    ("C113", "Suresh", "Kumar", 40, "Male", "Hyderabad", "2019-10-05"),

    ("C114", "Divya", "Iyer", 33, "Female", "Chennai", "2022-06-16"),

    ("C115", "Amit", "Joshi", 37, "Male", "Pune", "2020-03-28")

]

customer_df = spark.createDataFrame(customer_master_data,["customer_id", "first_name", "last_name", "age", "gender", "city", "customer_since"])
customer_df.show()

+-----------+----------+---------+---+------+---------+--------------+
|customer_id|first_name|last_name|age|gender|     city|customer_since|
+-----------+----------+---------+---+------+---------+--------------+
|       C101|       Ram|   Sharma| 28|  Male|Hyderabad|    2022-01-15|
|       C102|     Priya|    Reddy| 32|Female|Bengaluru|    2021-03-20|
|       C103|     Arjun|    Kumar| 35|  Male|  Chennai|    2020-06-10|
|       C104|     Sneha|    Patel| 29|Female|   Mumbai|    2023-02-05|
|       C105|     Rahul|    Verma| 41|  Male|    Delhi|    2019-08-18|
|       C106|    Anjali|     Shah| 27|Female|     Pune|    2022-11-12|
|       C107|    Vikram|    Singh| 38|  Male|  Kolkata|    2018-05-25|
|       C108|     Kavya|      Rao| 31|Female|Bengaluru|    2021-09-14|
|       C109|     Rohit|    Mehta| 45|  Male|  Chennai|    2017-04-22|
|       C110|      Neha|    Gupta| 34|Female|   Mumbai|    2020-12-01|
|       C111|     Kiran|    Reddy| 30|  Male|    Delhi|    2023-07-19|
|     

In [28]:
df_join=df.join(customer_df,on="customer_id",how="inner")
df_join.show()

+-----------+--------------+------------+----------------+-------+---------+----------------+----------+---------+---+------+---------+--------------+
|customer_id|transaction_id|account_type|transaction_type| amount|     city|transaction_date|first_name|last_name|age|gender|     city|customer_since|
+-----------+--------------+------------+----------------+-------+---------+----------------+----------+---------+---+------+---------+--------------+
|       C101|          1001|     Savings|         Deposit|50000.0|Hyderabad|      2026-01-05|       Ram|   Sharma| 28|  Male|Hyderabad|    2022-01-15|
|       C101|          1006|     Savings|        Transfer|15000.0|Hyderabad|      2026-01-18|       Ram|   Sharma| 28|  Male|Hyderabad|    2022-01-15|
|       C102|          1002|     Current|      Withdrawal|12000.0|Bengaluru|      2026-01-07|     Priya|    Reddy| 32|Female|Bengaluru|    2021-03-20|
|       C102|          1010|     Current|        Transfer|30000.0|Bengaluru|      2026-01-28| 

17. Repartition by account type and explain why.


In [ ]:
df.show()

In [ ]:
df_repartitioned=df.repartition("account_type")
df_repartitioned.show()

18. Build a final banking pipeline producing clean, enriched transactions.


In [96]:
count_city=df.groupBy("city").count().orderBy("count",ascending=False).first()["city"]

In [97]:
final_df=(df.filter(col("amount")>10000)\
            .withColumn("transaction_fee",col("amount")*0.01)\
            .withColumn("net_amount",col("amount")-col("transaction_fee"))\
            .withColumn("transaction_date",to_date("transaction_date"))\
            .withColumn("transaction_year",year("transaction_date"))\
            .withColumn("transaction_month",month("transaction_date"))\
            .withColumn("transaction_day",day("transaction_date"))\
            .withColumnRenamed("city","branch_city")\
            .drop("transaction_fee")\
            .dropDuplicates()\
            .orderBy("transaction_id",ascending=True)
            .fillna(count_city,"branch_city"))
# final_df.dropna()
final_df.show()
final_df.printSchema()


+--------------+-----------+------------+----------------+-------+-----------+----------------+----------+----------------+-----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|branch_city|transaction_date|net_amount|transaction_year|transaction_month|transaction_day|
+--------------+-----------+------------+----------------+-------+-----------+----------------+----------+----------------+-----------------+---------------+
|          1001|       C101|     Savings|         Deposit|50000.0|  Hyderabad|      2026-01-05|   49500.0|            2026|                1|              5|
|          1001|       C106|     Savings|         Deposit|50000.0|  Hyderabad|      2026-01-05|   49500.0|            2026|                1|              5|
|          1002|       C102|     Current|      Withdrawal|12000.0|  Bengaluru|      2026-01-07|   11880.0|            2026|                1|              7|
|          1003|       C103|     Savings|        Tra

In [87]:
final_df1=(df.filter(col("transaction_type")=="Withdrawal")\
            .withColumn("transaction_fee",col("amount")*0.01)\
            .withColumn("net_amount",col("amount")-col("transaction_fee"))\
            .withColumn("transaction_date",to_date("transaction_date"))\
            .withColumn("transaction_year",year("transaction_date"))\
            .withColumn("transaction_month",month("transaction_date"))\
            .withColumn("transaction_day",day("transaction_date"))\
            .withColumnRenamed("city","branch_city")\
            .drop("transaction_fee")\
            .dropDuplicates()\
            .orderBy("transaction_id",ascending=True))
final_df1.printSchema()
final_df1.dropna()
final_df1.show()


root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- branch_city: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- transaction_year: integer (nullable = true)
 |-- transaction_month: integer (nullable = true)
 |-- transaction_day: integer (nullable = true)

+--------------+-----------+------------+----------------+-------+-----------+----------------+----------+----------------+-----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type| amount|branch_city|transaction_date|net_amount|transaction_year|transaction_month|transaction_day|
+--------------+-----------+------------+----------------+-------+-----------+----------------+----------+----------------+-----------------+---------------+
|          1002|       C1

**Challenge:** Classify the transformations you used as narrow or wide and identify where a shuffle is likely to occur.



Narrow Transformation such as filter,withColumn,withColumnRenamed and drop operates independently on same partition but the wide transformation such as groupBy,OrderBy,dropDuplicates are dependent on each other and have to move between partitions where the actual suffle happens